In [ ]:
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

def coords():

    files = glob.glob(f'{results}/catalogs/*_f090w_dropouts_aper_*.fits')
    #files = glob.glob(f'{data}/catalogs/photometryCatalog_*_feb2026.fits')

    for file in files:

        hdul = fits.open(file)

        ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

        fig, ax = plt.subplots()

        ax.scatter(ra, dec, c='black', marker='o')

        ax.set_xlabel('Right ascension (deg.)')
        ax.set_ylabel('Declination (deg.)')

        ax.xaxis.set_inverted(True)

        fig.savefig(f'{figs}/{os.path.basename(file)}.png', bbox_inches='tight', dpi=200)

def footprints():

    #files = glob.glob(f'{results}/catalogs/*_f090w_dropouts_aper_0.fits')
    files = glob.glob(f'{data}/catalogs/photometryCatalog_*_feb2026.fits')

    filters = ['f090w','f115w','f150w','f200w','f277w','f356w','f410m','f444w']

    for file in files:

        hdul = fits.open(file)

        # Get the filters in the catalog
        #filters = [col.split('_')[0] for col in hdul[1].columns.names if col.endswith('_tot_0')]

        ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

        print(ra, dec)

        for filter in filters:

            #ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

            mask = ~np.isnan(hdul[1].data[f'{filter}_tot_0'])

            fig, ax = plt.subplots()

            ax.scatter(ra[mask], dec[mask], c='black', marker='o')

            ax.set_xlabel('Right ascension (deg.)')
            ax.set_ylabel('Declination (deg.)')

            ax.xaxis.set_inverted(True)

            at = AnchoredText(filter, loc='upper right')
            ax.add_artist(at)

            #fig.savefig(f'{figs}/{os.path.basename(file)}.png', bbox_inches='tight', dpi=200)

def cutouts():

    files = glob.glob(f'{results}/catalogs/*_f070w_dropouts_init.fits')

    for file in files:

        hdul = fits.open(file)

        ids = hdul[1].data['ID']
        ra, dec = hdul[1].data['RA'], hdul[1].data['DEC']

        for i, _ in enumerate(ra):

            cutout(ra[i], dec[i], filters=['f435w', 'f606w','f070w','f814w','f090w-clear','f150w-clear'], id=ids[i], save=True, dir='figs/f070w_dropouts_init_cutouts')
            #cutout(ra[i], dec[i], id=ids[i], save=True, dir='figs/f070w_dropouts_init_cutouts')

def check_dja():

    '''
    Check for extant spectroscopic observations of the F090W dropouts in the DJA catalog
    '''

    # Get the F090W dropout catalogs
    files = glob.glob(f'{results}/catalogs/*_f090w_dropouts_aper_*.fits')

    # For each catalog file
    for file in files:

        # Open the HDU list of the catalog
        hdul = fits.open(file)

        # Get the coordinates of the sources as a SkyCoord object
        coords = SkyCoord(ra=hdul[1].data['RA'] * u.deg, dec=hdul[1].data['DEC'] * u.deg)

        # Check for coordinate matches in the spectroscopic DJA catalog
        check(coords)